In [1]:
import pandas as pd
accounts = pd.read_csv(r"C:\Users\Vivek\OneDrive\Desktop\Indore\Data Analytics\Tableau\ravenstack_accounts.csv",encoding="latin1")
subs = pd.read_csv(r"C:\Users\Vivek\OneDrive\Desktop\Indore\Data Analytics\Tableau\ravenstack_subscriptions.csv",encoding="latin1")
tickets = pd.read_csv(r"C:\Users\Vivek\OneDrive\Desktop\Indore\Data Analytics\Tableau\ravenstack_support_tickets.csv",encoding="latin1")
usage = pd.read_csv(r"C:\Users\Vivek\OneDrive\Desktop\Indore\Data Analytics\Tableau\ravenstack_feature_usage.csv",encoding="latin1")
churn = pd.read_csv(r"C:\Users\Vivek\OneDrive\Desktop\Indore\Data Analytics\Tableau\ravenstack_churn_events.csv",encoding="latin1")


AGGREGATE USAGE: Find total usage per subscription

In [2]:
usage_summary = usage.groupby('subscription_id').agg(
    total_usage_count=('usage_count', 'sum'),
    avg_usage_duration=('usage_duration_secs', 'mean'),
    total_errors=('error_count', 'sum')
).reset_index()

 AGGREGATE TICKETS: Count tickets per account

In [3]:
ticket_summary = tickets.groupby('account_id').agg(
    ticket_count=('ticket_id', 'count'),
    avg_satisfaction=('satisfaction_score', 'mean')
).reset_index()

MASTER JOIN: Start with Subscriptions as the base

In [4]:
master_df = subs.merge(accounts, on='account_id', how='left', suffixes=('', '_acc'))
master_df = master_df.merge(usage_summary, on='subscription_id', how='left')
master_df = master_df.merge(ticket_summary, on='account_id', how='left')
master_df = master_df.merge(churn[['account_id', 'reason_code', 'feedback_text']], on='account_id', how='left')

CLEANING & LOGIC

In [5]:
master_df['ticket_count'] = master_df['ticket_count'].fillna(0)
master_df['total_usage_count'] = master_df['total_usage_count'].fillna(0)
master_df['avg_satisfaction'] = master_df['avg_satisfaction'].fillna(master_df['avg_satisfaction'].mean())


 Create the health flag for Tableau

In [6]:
master_df['Account_Health'] = 'Stable'
master_df.loc[(master_df['ticket_count'] > 3) & (master_df['total_usage_count'] < 50), 'Account_Health'] = 'High Risk'

In [7]:
master_df.to_csv('RavenStack_Master_Analysis.csv')
print("--- PROCESS COMPLETE ---")
print(f"File Saved: RavenStack_Master_Analysis.csv")
print(f"Total Records: {len(master_df)}")

--- PROCESS COMPLETE ---
File Saved: RavenStack_Master_Analysis.csv
Total Records: 7429


In [8]:
print(master_df.columns)

Index(['subscription_id', 'account_id', 'start_date', 'end_date', 'plan_tier',
       'seats', 'mrr_amount', 'arr_amount', 'is_trial', 'upgrade_flag',
       'downgrade_flag', 'churn_flag', 'billing_frequency', 'auto_renew_flag',
       'account_name', 'industry', 'country', 'signup_date', 'referral_source',
       'plan_tier_acc', 'seats_acc', 'is_trial_acc', 'churn_flag_acc',
       'total_usage_count', 'avg_usage_duration', 'total_errors',
       'ticket_count', 'avg_satisfaction', 'reason_code', 'feedback_text',
       'Account_Health'],
      dtype='str')
